# 🧵 Python Multithreading Guide

> A comprehensive reference guide for understanding threading, locks, and synchronization patterns in Python

---

## 📑 Table of Contents

| # | Topic | Description |
|---|-------|-------------|
| 1 | [**🧵 Threads**](#threads) | Basics of threading for I/O-bound tasks |
| 2 | [**⚙️ ThreadPoolExecutor**](#threadpoolexecutor) | Managing multiple threads efficiently |
| 3 | [**⚠️ Race Conditions**](#race-conditions) | Understanding shared resource conflicts |
| 4 | [**🔒 Locks & Synchronization**](#locks--synchronization) | Preventing race conditions with locks |
| 5 | [**💀 Deadlocks**](#deadlocks) | Recognizing and avoiding deadlock scenarios |
| 6 | [**🔄 Producer-Consumer Pattern**](#producer-consumer-pattern) | Real-world synchronization example |

---

# 🧵 Threads

- Useful for I/O-bound tasks
- Due to the Global Interpreter Lock (GIL), threads can only run sequentially in Python, though different threads can run on different processor cores
- For CPU-bound programs, prefer `multiprocessing`
- To start a thread: `threading.Thread(target=target_function, daemon=optional_bool)`

In [ ]:
# first implementation

import logging
import threading
import time


def thread_function(name):
    logging.info("Thread %s: starting", name)
    time.sleep(2)
    logging.info("Thread %s: finishing", name)

if __name__ == "__main__":
    format = "%(asctime)s: %(message)s"
    logging.basicConfig(format=format, level=logging.INFO,
                        datefmt="%H:%M:%S")

    logging.info("Main    : before creating thread")
    x = threading.Thread(target=thread_function, args=(1,))
    logging.info("Main    : before running thread")
    x.start()
    logging.info("Main    : wait for the thread to finish")
    x.join()
    logging.info("Main    : all done")

## Important Notes

- We are not yet joining the thread in the example above
- `if __name__ == "__main__":` is a protection mechanism that prevents this code from executing on import. This is especially important in multithreading/multiprocessing because some operating systems re-import the main module when spawning new threads or processes. Without this guard, infinite recursion could occur
- `daemon`: A daemon thread is terminated when the main program exits
- `.join()`: Forces the main thread to wait for the target thread to finish before continuing

# ⚙️ ThreadPoolExecutor

Instead of managing individual threads manually, it's better to use a `ThreadPoolExecutor`:
- The recommended approach is to create it as a context manager using the `with` statement for proper resource management
- The context manager automatically calls `.join()` on all threads in the pool when exiting

In [ ]:
import logging
import threading
import time
import concurrent.futures

def thread_function(name):
    logging.info("Thread %s: starting", name)
    time.sleep(2)
    logging.info("Thread %s: finishing", name)

if __name__ == "__main__":
    format = "%(asctime)s: %(message)s"
    logging.basicConfig(format=format, level=logging.INFO,
                        datefmt="%H:%M:%S")

    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
        executor.map(thread_function, range(3))

# ⚠️ Race Conditions

A race condition occurs when multiple threads compete for access to shared resources or data, leading to unpredictable behavior.

The example below demonstrates this problem:

In [ ]:
#Fake database example below

import logging
import threading
import concurrent.futures
import time

class FakeDatabase:
    def __init__(self):
        self.value = 0

    def update(self, name):
        logging.info("Thread %s: starting update", name)
        local_copy = self.value
        local_copy += 1 #local-copy here is local to the function, so thread-safe
        time.sleep(0.1)
        self.value = local_copy #this is a shared resource without any lock, will get updated by both threads
        logging.info("Thread %s: finishing update", name)

if __name__ == "__main__":
    format = "%(asctime)s: %(message)s"
    logging.basicConfig(format=format, level=logging.INFO,
                        datefmt="%H:%M:%S")

    database = FakeDatabase()
    logging.info("Testing update. Starting value is %d.", database.value)
    with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
        for index in range(2):
            executor.submit(database.update, index)
    logging.info("Testing update. Ending value is %d.", database.value)


# 🔒 Locks & Synchronization

To prevent race conditions, we use a `Lock` to ensure only one thread can access the critical section at a time. Below we add comprehensive logging to see exactly how locks are acquired and released:

In [ ]:
#let's add a Lock and understand the race condition fix

import logging
import threading
import concurrent.futures
import time

class FakeDatabase:
    def __init__(self):
        self.value = 0
        self._lock = threading.Lock()

    def locked_update(self, name):
        logging.info("Thread %s: starting update", name)
        logging.debug("Thread %s about to lock", name)
        with self._lock: #again, using context manager to be sure we release the lock in the end, like
            # we did wit the ThreadPoolExecutor below
            logging.debug("Thread %s has lock", name)
            local_copy = self.value
            local_copy += 1
            time.sleep(0.1)
            self.value = local_copy
            logging.debug("Thread %s about to release lock", name)
        logging.debug("Thread %s after release", name)
        logging.info("Thread %s: finishing update", name)

if __name__ == "__main__":
    format = "%(asctime)s: %(message)s"
    logging.basicConfig(format=format, level=logging.INFO,
                        datefmt="%H:%M:%S")
    logging.getLogger().setLevel(logging.DEBUG)

    database = FakeDatabase()
    logging.info("Testing update. Starting value is %d.", database.value)
    with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
        for index in range(2):
            executor.submit(database.locked_update, index)
    logging.info("Testing update. Ending value is %d.", database.value)

# 💀 Deadlocks

A deadlock occurs when two or more threads wait indefinitely for each other to release locks:

```
- Thread 1 acquires lock_a, then waits for lock_b
- Thread 2 acquires lock_b, then waits for lock_a
→ Both threads are now stuck, waiting for each other
```

## Common Causes and Solutions

**1. Implementation Bug — Lock is not released**
- Solution: Always use a context manager (like `with self._lock:`) to guarantee the lock is released

**2. Design Issue — Recursive locking**
- Problem: A utility function that needs to be called by functions that already hold the lock. If the utility function also acquires the same lock, the thread will deadlock waiting for itself
- Solution: Use `RLock` (Reentrant Lock), which allows the same thread to acquire it multiple times without deadlock

# 🔄 Producer-Consumer Pattern

A practical example of using locks and multithreading together. In this pattern:
- A **producer** thread fetches random messages from a fake network
- A **consumer** thread processes and "saves" these messages
- Messages arrive in bursts and are synchronized using locks

In [ ]:
import random

SENTINEL = object()

def producer(pipeline):
    """Simulate fetching a message from the network."""
    for index in range(10):
        message = random.randint(1, 101)
        logging.info("Producer got message: %s", message)
        pipeline.set_message(message, "Producer")

    # Send a sentinel message to signal the consumer that we're done
    pipeline.set_message(SENTINEL, "Producer")

def consumer(pipeline):
    """Simulate saving a message to a database."""
    message = 0
    while message is not SENTINEL:
        message = pipeline.get_message("Consumer")
        if message is not SENTINEL:
            logging.info("Consumer storing message: %s", message)

class Pipeline:
    """
    A single-element pipeline to synchronize producer and consumer threads.
    Uses locks to ensure thread-safe access to the shared message.
    """
    def __init__(self):
        self.message = 0
        self.producer_lock = threading.Lock()
        self.consumer_lock = threading.Lock()
        # Start with consumer lock acquired so producer goes first
        self.consumer_lock.acquire()

    def get_message(self, name):
        logging.debug("%s: about to acquire consumer lock", name)
        self.consumer_lock.acquire()
        logging.debug("%s: acquired consumer lock", name)
        message = self.message
        logging.debug("%s: about to release producer lock", name)
        self.producer_lock.release()
        logging.debug("%s: released producer lock", name)
        return message

    def set_message(self, message, name):
        logging.debug("%s: about to acquire producer lock", name)
        self.producer_lock.acquire()
        logging.debug("%s: acquired producer lock", name)
        self.message = message
        logging.debug("%s: about to release consumer lock", name)
        self.consumer_lock.release()
        logging.debug("%s: released consumer lock", name)


if __name__ == "__main__":
    format = "%(asctime)s: %(message)s"
    logging.basicConfig(format=format, level=logging.INFO,
                        datefmt="%H:%M:%S")
    # logging.getLogger().setLevel(logging.DEBUG)

    pipeline = Pipeline()
    with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
        executor.submit(producer, pipeline)
        executor.submit(consumer, pipeline)